# Logistic Regression Lab — NetFlow Traffic Classification

## Objective

Use logistic regression to classify network flows as **Normal**, **Port Scan**, **DDoS**, or **Data Exfiltration** based on NetFlow features:
- **Duration** — how long the flow lasted
- **Bytes sent/received** — volume of data transferred
- **Packets sent/received** — number of packets
- **Destination port** — which service was targeted
- **Protocol** — TCP, UDP, or ICMP
- **Flags** — TCP flag distribution (SYN, ACK, RST, FIN ratios)

## Learning Goals

1. Understand why linear regression is unsuitable for classification
2. Learn how the sigmoid function maps scores to probabilities
3. Build a binary classifier (normal vs attack)
4. Extend to multi-class classification (4 traffic types)
5. Evaluate with confusion matrix, precision, recall, and F1
6. Interpret coefficients to understand which features signal attacks

## 1. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, RocCurveDisplay
)
from sklearn.pipeline import Pipeline

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
np.random.seed(42)

print("Libraries loaded successfully.")

## 2. Generate Synthetic NetFlow Dataset

We simulate realistic NetFlow records for 4 traffic classes:

| Class | Characteristics |
|:------|:---------------|
| **Normal** | Moderate duration, balanced bytes/packets, common ports (80, 443, 53) |
| **Port Scan** | Very short flows, tiny packets, many distinct destination ports, high SYN ratio |
| **DDoS** | Short duration, huge packet count, high bytes, SYN flood or UDP amplification |
| **Data Exfiltration** | Long duration, large outbound bytes, low packet rate, uncommon ports |

In [ ]:
def generate_netflow_data(n_per_class=1000):
    """Generate synthetic NetFlow records for 4 traffic classes."""
    records = []
    
    # --- Normal Traffic ---
    for _ in range(n_per_class):
        duration = np.random.exponential(30) + 0.5  # 0.5s to ~120s
        packets_sent = int(np.random.uniform(5, 200))
        packets_recv = int(np.random.uniform(5, 300))
        bytes_sent = packets_sent * np.random.uniform(100, 1400)
        bytes_recv = packets_recv * np.random.uniform(100, 1400)
        dst_port = np.random.choice([80, 443, 53, 22, 25, 110, 993, 8080, 3306])
        protocol = np.random.choice([6, 17, 6, 6], p=[0.25, 0.25, 0.25, 0.25])  # TCP heavy
        syn_ratio = np.random.uniform(0.01, 0.15)
        rst_ratio = np.random.uniform(0.0, 0.05)
        records.append([duration, bytes_sent, bytes_recv, packets_sent, packets_recv,
                       dst_port, protocol, syn_ratio, rst_ratio, "Normal"])
    
    # --- Port Scan ---
    for _ in range(n_per_class):
        duration = np.random.exponential(0.5) + 0.01  # very short
        packets_sent = int(np.random.uniform(1, 5))
        packets_recv = int(np.random.uniform(0, 3))
        bytes_sent = packets_sent * np.random.uniform(40, 80)  # small SYN packets
        bytes_recv = packets_recv * np.random.uniform(40, 80)
        dst_port = np.random.randint(1, 65535)  # random ports
        protocol = 6  # TCP (SYN scan)
        syn_ratio = np.random.uniform(0.7, 1.0)  # mostly SYN
        rst_ratio = np.random.uniform(0.3, 0.8)  # many RST responses
        records.append([duration, bytes_sent, bytes_recv, packets_sent, packets_recv,
                       dst_port, protocol, syn_ratio, rst_ratio, "Port Scan"])
    
    # --- DDoS ---
    for _ in range(n_per_class):
        duration = np.random.exponential(5) + 0.1  # short bursts
        packets_sent = int(np.random.uniform(500, 50000))  # massive packet count
        packets_recv = int(np.random.uniform(0, 100))  # minimal response
        bytes_sent = packets_sent * np.random.uniform(40, 200)  # small packets, high volume
        bytes_recv = packets_recv * np.random.uniform(40, 100)
        dst_port = np.random.choice([80, 443, 53, 123, 161])  # common DDoS targets
        protocol = np.random.choice([6, 17], p=[0.5, 0.5])  # TCP SYN flood or UDP
        syn_ratio = np.random.uniform(0.6, 0.95) if protocol == 6 else np.random.uniform(0.0, 0.1)
        rst_ratio = np.random.uniform(0.0, 0.1)
        records.append([duration, bytes_sent, bytes_recv, packets_sent, packets_recv,
                       dst_port, protocol, syn_ratio, rst_ratio, "DDoS"])
    
    # --- Data Exfiltration ---
    for _ in range(n_per_class):
        duration = np.random.uniform(60, 3600)  # long-lived
        packets_sent = int(np.random.uniform(100, 2000))
        packets_recv = int(np.random.uniform(10, 100))  # mostly outbound
        bytes_sent = packets_sent * np.random.uniform(1000, 1460)  # large payloads out
        bytes_recv = packets_recv * np.random.uniform(40, 200)  # small ACKs back
        dst_port = np.random.choice([443, 8443, 4444, 53, 8080, 9001])  # covert channels
        protocol = 6  # TCP
        syn_ratio = np.random.uniform(0.001, 0.02)  # established connection
        rst_ratio = np.random.uniform(0.0, 0.01)
        records.append([duration, bytes_sent, bytes_recv, packets_sent, packets_recv,
                       dst_port, protocol, syn_ratio, rst_ratio, "Exfiltration"])
    
    columns = ["duration_s", "bytes_sent", "bytes_recv", "packets_sent", "packets_recv",
              "dst_port", "protocol", "syn_ratio", "rst_ratio", "label"]
    return pd.DataFrame(records, columns=columns)

df = generate_netflow_data(n_per_class=1000)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

print(f"Dataset shape: {df.shape}")
print(f"\nClass distribution:")
print(df["label"].value_counts())
df.head(10)

## 3. Feature Engineering

Derive additional features that are commonly used in network intrusion detection.

In [ ]:
# Derived features
df["total_bytes"] = df["bytes_sent"] + df["bytes_recv"]
df["total_packets"] = df["packets_sent"] + df["packets_recv"]
df["bytes_per_packet"] = df["total_bytes"] / df["total_packets"].clip(lower=1)
df["packets_per_second"] = df["total_packets"] / df["duration_s"].clip(lower=0.001)
df["bytes_per_second"] = df["total_bytes"] / df["duration_s"].clip(lower=0.001)
df["sent_recv_byte_ratio"] = df["bytes_sent"] / df["bytes_recv"].clip(lower=1)
df["sent_recv_pkt_ratio"] = df["packets_sent"] / df["packets_recv"].clip(lower=1)

# Log-transform skewed features
df["log_duration"] = np.log1p(df["duration_s"])
df["log_bytes_per_second"] = np.log1p(df["bytes_per_second"])
df["log_packets_per_second"] = np.log1p(df["packets_per_second"])

print("Engineered features added.")
print(f"Total features: {df.shape[1] - 1}")
df[["label", "bytes_per_packet", "packets_per_second", "sent_recv_byte_ratio", "log_duration"]].head(10)

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Summary statistics by class
summary = df.groupby("label")[["duration_s", "total_bytes", "total_packets",
                                "bytes_per_packet", "packets_per_second",
                                "syn_ratio", "rst_ratio"]].median().round(2)
print("=== Median Values by Traffic Class ===")
summary

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

features_to_plot = [
    ("log_duration", "Log Duration (s)"),
    ("log_packets_per_second", "Log Packets/sec"),
    ("bytes_per_packet", "Bytes per Packet"),
    ("syn_ratio", "SYN Ratio"),
    ("rst_ratio", "RST Ratio"),
    ("sent_recv_byte_ratio", "Sent/Recv Byte Ratio"),
]

colors = {"Normal": "steelblue", "Port Scan": "coral",
          "DDoS": "mediumpurple", "Exfiltration": "seagreen"}

for ax, (feat, title) in zip(axes.flat, features_to_plot):
    for label, color in colors.items():
        subset = df[df["label"] == label][feat]
        ax.hist(subset, bins=30, alpha=0.5, label=label, color=color, density=True)
    ax.set_xlabel(title)
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)
    ax.set_title(title)

plt.suptitle("Feature Distributions by Traffic Class", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# 2D scatter: key discriminating features
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color in colors.items():
    subset = df[df["label"] == label]
    axes[0].scatter(subset["log_packets_per_second"], subset["syn_ratio"],
                   alpha=0.3, s=10, color=color, label=label)
    axes[1].scatter(subset["log_duration"], subset["sent_recv_byte_ratio"].clip(upper=50),
                   alpha=0.3, s=10, color=color, label=label)

axes[0].set_xlabel("Log Packets per Second")
axes[0].set_ylabel("SYN Ratio")
axes[0].set_title("Packets Rate vs SYN Ratio")
axes[0].legend()

axes[1].set_xlabel("Log Duration")
axes[1].set_ylabel("Sent/Recv Byte Ratio (clipped at 50)")
axes[1].set_title("Duration vs Data Direction")
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Why Not Linear Regression for Classification?

Let's quickly demonstrate why linear regression fails for classification tasks.

In [ ]:
# Binary simplification: Normal (0) vs Attack (1)
df["is_attack"] = (df["label"] != "Normal").astype(int)

# Fit linear regression on a single feature for illustration
from sklearn.linear_model import LinearRegression

x_demo = df["syn_ratio"].values.reshape(-1, 1)
y_demo = df["is_attack"].values

lr = LinearRegression()
lr.fit(x_demo, y_demo)

x_range = np.linspace(0, 1, 200).reshape(-1, 1)
y_lr = lr.predict(x_range)

# Fit logistic regression for comparison
log_reg = LogisticRegression()
log_reg.fit(x_demo, y_demo)
y_logistic = log_reg.predict_proba(x_range)[:, 1]

# Plot
plt.figure(figsize=(10, 5))
plt.scatter(df["syn_ratio"], df["is_attack"], alpha=0.1, s=5, color="gray", label="Data points")
plt.plot(x_range, y_lr, color="coral", linewidth=2, linestyle="--", label="Linear Regression")
plt.plot(x_range, y_logistic, color="steelblue", linewidth=2, label="Logistic Regression")
plt.axhline(y=0.5, color="black", linestyle=":", alpha=0.5, label="Decision boundary (0.5)")
plt.xlabel("SYN Ratio")
plt.ylabel("P(Attack)")
plt.title("Why Linear Regression Fails for Classification")
plt.ylim(-0.2, 1.3)
plt.legend()
plt.tight_layout()
plt.show()

print("Problems with Linear Regression for classification:")
print("  • Outputs can be < 0 or > 1 (not valid probabilities)")
print("  • No natural threshold interpretation")
print("  • Sensitive to outliers in feature values")
print("\nLogistic Regression fixes these with the sigmoid function:")
print("  σ(z) = 1 / (1 + e^(-z))  →  always outputs [0, 1]")

## 6. Data Preparation

In [ ]:
# Select features for classification
feature_cols = [
    "log_duration", "bytes_per_packet", "log_packets_per_second",
    "log_bytes_per_second", "syn_ratio", "rst_ratio",
    "sent_recv_byte_ratio", "sent_recv_pkt_ratio", "protocol"
]

X = df[feature_cols].copy()
y_binary = df["is_attack"]          # Binary: 0=Normal, 1=Attack
y_multi = df["label"]               # Multi-class: 4 traffic types

# Train/test split (stratified to maintain class balance)
X_train, X_test, y_bin_train, y_bin_test = train_test_split(
    X, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)
_, _, y_multi_train, y_multi_test = train_test_split(
    X, y_multi, test_size=0.2, random_state=42, stratify=y_multi
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples:     {len(X_test)}")
print(f"\nFeatures used ({len(feature_cols)}):")
for f in feature_cols:
    print(f"  • {f}")
print(f"\nBinary class balance (train): {y_bin_train.value_counts().to_dict()}")
print(f"Multi-class balance (train):  {y_multi_train.value_counts().to_dict()}")

## 7. Binary Classification: Normal vs Attack

### The Sigmoid Function

$$P(\text{Attack}) = \sigma(z) = \frac{1}{1 + e^{-z}}$$

where $z = w_1 \cdot \text{log\_duration} + w_2 \cdot \text{bytes\_per\_packet} + \dots + b$

If $P(\text{Attack}) \geq 0.5$, classify as **Attack**.

In [ ]:
# Build pipeline: Scale + Logistic Regression
pipe_binary = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=1000, random_state=42))
])

pipe_binary.fit(X_train, y_bin_train)

# Predictions
y_bin_pred = pipe_binary.predict(X_test)
y_bin_proba = pipe_binary.predict_proba(X_test)[:, 1]

print("=== Binary Classification: Normal vs Attack ===")
print(f"\nAccuracy: {pipe_binary.score(X_test, y_bin_test):.4f}")
print("\nClassification Report:")
print(classification_report(y_bin_test, y_bin_pred,
                           target_names=["Normal", "Attack"]))

In [ ]:
# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ConfusionMatrixDisplay.from_predictions(
    y_bin_test, y_bin_pred,
    display_labels=["Normal", "Attack"],
    cmap="Blues", ax=axes[0]
)
axes[0].set_title("Binary Confusion Matrix")

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_bin_test, y_bin_proba)
roc_auc = auc(fpr, tpr)

axes[1].plot(fpr, tpr, color="steelblue", linewidth=2, label=f"ROC (AUC = {roc_auc:.3f})")
axes[1].plot([0, 1], [0, 1], "k--", linewidth=1, label="Random")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve — Normal vs Attack")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nAUC Score: {roc_auc:.4f}")
print("\nInterpretation:")
print("  • AUC = 1.0 → perfect separation")
print("  • AUC = 0.5 → random guessing")
print(f"  • Our model: {roc_auc:.3f} → {'excellent' if roc_auc > 0.95 else 'good'} discrimination")

## 8. Interpreting Binary Model Coefficients

In logistic regression, coefficients tell us how each feature affects the **log-odds** of being an attack.

In [ ]:
# Extract coefficients (from standardised features)
coefs = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": pipe_binary.named_steps["logreg"].coef_[0],
    "|Coefficient|": np.abs(pipe_binary.named_steps["logreg"].coef_[0])
}).sort_values("|Coefficient|", ascending=False)

print("=== Feature Coefficients (Standardised) ===")
print("Positive = increases P(Attack), Negative = decreases P(Attack)\n")
print(coefs.to_string(index=False))

# Bar chart
plt.figure(figsize=(10, 5))
colors = ["coral" if c > 0 else "steelblue" for c in coefs["Coefficient"]]
plt.barh(coefs["Feature"], coefs["Coefficient"], color=colors)
plt.axvline(x=0, color="black", linewidth=0.5)
plt.xlabel("Coefficient (effect on log-odds of Attack)")
plt.title("Feature Importance — Binary Classification")
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  • High SYN ratio → strong attack signal (port scans, SYN floods)")
print("  • High packets/sec → DDoS indicator")
print("  • Long duration → could be exfiltration (but also normal downloads)")
print("  • Large bytes/packet with high outbound ratio → exfiltration")

## 9. Multi-Class Classification (4 Traffic Types)

Logistic regression extends to multiple classes using **One-vs-Rest (OvR)** or **Softmax (multinomial)**:

- **OvR**: Train one binary classifier per class (4 classifiers)
- **Multinomial**: Single model, softmax output layer

$$P(\text{class}_k) = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}$$

In [ ]:
# Multi-class logistic regression (multinomial with softmax)
pipe_multi = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        solver="lbfgs",
        max_iter=1000,
        random_state=42
    ))
])

pipe_multi.fit(X_train, y_multi_train)

# Predictions
y_multi_pred = pipe_multi.predict(X_test)

print("=== Multi-Class Classification (4 Traffic Types) ===")
print(f"\nAccuracy: {pipe_multi.score(X_test, y_multi_test):.4f}")
print("\nClassification Report:")
print(classification_report(y_multi_test, y_multi_pred))

In [ ]:
# Multi-class confusion matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_multi_test, y_multi_pred, labels=["Normal", "Port Scan", "DDoS", "Exfiltration"])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
           xticklabels=["Normal", "Port Scan", "DDoS", "Exfil"],
           yticklabels=["Normal", "Port Scan", "DDoS", "Exfil"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Multi-Class Confusion Matrix")
plt.tight_layout()
plt.show()

print("\nReading the confusion matrix:")
print("  • Diagonal = correct predictions")
print("  • Off-diagonal = misclassifications")
print("  • Look for patterns: which classes are confused with each other?")

## 10. Multi-Class Coefficient Analysis

In multinomial logistic regression, each class has its **own set of coefficients**.

In [ ]:
# Coefficient heatmap
classes = pipe_multi.named_steps["logreg"].classes_
coef_matrix = pd.DataFrame(
    pipe_multi.named_steps["logreg"].coef_,
    index=classes,
    columns=feature_cols
)

plt.figure(figsize=(12, 5))
sns.heatmap(coef_matrix, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
           xticklabels=[f.replace("_", "\n") for f in feature_cols])
plt.title("Logistic Regression Coefficients per Class")
plt.ylabel("Traffic Class")
plt.tight_layout()
plt.show()

print("\nInterpretation (positive = feature supports this class):")
print("")
for cls in classes:
    top_features = coef_matrix.loc[cls].nlargest(3)
    print(f"  {cls}:")
    for feat, val in top_features.items():
        print(f"    • {feat}: {val:+.2f}")
    print()

## 11. Probability Outputs and Confidence

Logistic regression provides **calibrated probabilities** — useful for setting alert thresholds.

In [ ]:
# Get probability predictions
probas = pipe_multi.predict_proba(X_test)
proba_df = pd.DataFrame(probas, columns=classes)
proba_df["predicted"] = y_multi_pred
proba_df["actual"] = y_multi_test.values
proba_df["confidence"] = probas.max(axis=1)
proba_df["correct"] = (proba_df["predicted"] == proba_df["actual"])

print("=== Prediction Confidence ===")
print(f"\nMean confidence (correct predictions): {proba_df[proba_df['correct']]['confidence'].mean():.3f}")
print(f"Mean confidence (wrong predictions):   {proba_df[~proba_df['correct']]['confidence'].mean():.3f}")

# Show some uncertain predictions
uncertain = proba_df.nsmallest(10, "confidence")
print("\nMost uncertain predictions (lowest confidence):")
print(uncertain[["actual", "predicted", "confidence"]].to_string(index=False))

In [ ]:
# Confidence distribution by correctness
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(proba_df[proba_df["correct"]]["confidence"], bins=30,
            alpha=0.7, color="seagreen", label="Correct", density=True)
axes[0].hist(proba_df[~proba_df["correct"]]["confidence"], bins=30,
            alpha=0.7, color="coral", label="Wrong", density=True)
axes[0].set_xlabel("Prediction Confidence")
axes[0].set_ylabel("Density")
axes[0].set_title("Confidence Distribution")
axes[0].legend()

# Accuracy at different confidence thresholds
thresholds = np.arange(0.3, 1.0, 0.05)
accuracies = []
coverages = []
for t in thresholds:
    mask = proba_df["confidence"] >= t
    if mask.sum() > 0:
        accuracies.append(proba_df[mask]["correct"].mean())
        coverages.append(mask.mean())
    else:
        accuracies.append(np.nan)
        coverages.append(0)

ax2 = axes[1]
ax2.plot(thresholds, accuracies, color="steelblue", linewidth=2, label="Accuracy")
ax2_twin = ax2.twinx()
ax2_twin.plot(thresholds, coverages, color="coral", linewidth=2, linestyle="--", label="Coverage")
ax2.set_xlabel("Confidence Threshold")
ax2.set_ylabel("Accuracy", color="steelblue")
ax2_twin.set_ylabel("Coverage (% classified)", color="coral")
ax2.set_title("Accuracy vs Coverage Trade-off")
ax2.legend(loc="lower left")
ax2_twin.legend(loc="lower right")

plt.tight_layout()
plt.show()

print("\nPractical use: Set a confidence threshold for your SOC alerts.")
print("  • High threshold (0.9+): fewer alerts, but very precise")
print("  • Low threshold (0.5): catch more attacks, but more false positives")

## 12. Cross-Validation

In [ ]:
# 5-fold cross-validation for both models
cv_binary = cross_val_score(
    Pipeline([("scaler", StandardScaler()),
              ("logreg", LogisticRegression(max_iter=1000, random_state=42))]),
    X, y_binary, cv=5, scoring="f1"
)

cv_multi = cross_val_score(
    Pipeline([("scaler", StandardScaler()),
              ("logreg", LogisticRegression(solver="lbfgs",
                                           max_iter=1000, random_state=42))]),
    X, y_multi, cv=5, scoring="f1_weighted"
)

print("=== 5-Fold Cross-Validation ===")
print(f"\nBinary (F1):           {cv_binary.mean():.4f} ± {cv_binary.std():.4f}")
print(f"  Folds: {[f'{s:.4f}' for s in cv_binary]}")
print(f"\nMulti-class (F1-wtd):  {cv_multi.mean():.4f} ± {cv_multi.std():.4f}")
print(f"  Folds: {[f'{s:.4f}' for s in cv_multi]}")
print(f"\n→ Low variance across folds = stable model.")

## 13. Threshold Tuning for Security Operations

In network security, **recall** (catching all attacks) often matters more than precision. Let's tune the decision threshold.

In [ ]:
from sklearn.metrics import precision_recall_curve, f1_score

# Precision-Recall trade-off for binary classifier
precision_vals, recall_vals, thresholds_pr = precision_recall_curve(
    y_bin_test, y_bin_proba
)

# Find F1 for different thresholds
f1_scores = []
test_thresholds = np.arange(0.1, 0.9, 0.01)
for t in test_thresholds:
    y_pred_t = (y_bin_proba >= t).astype(int)
    f1_scores.append(f1_score(y_bin_test, y_pred_t))

best_threshold = test_thresholds[np.argmax(f1_scores)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Precision-Recall curve
axes[0].plot(recall_vals, precision_vals, color="steelblue", linewidth=2)
axes[0].set_xlabel("Recall (Sensitivity)")
axes[0].set_ylabel("Precision")
axes[0].set_title("Precision-Recall Curve")
axes[0].axhline(y=0.75, color="gray", linestyle=":", alpha=0.5)

# F1 vs threshold
axes[1].plot(test_thresholds, f1_scores, color="mediumpurple", linewidth=2)
axes[1].axvline(x=0.5, color="gray", linestyle="--", alpha=0.5, label="Default (0.5)")
axes[1].axvline(x=best_threshold, color="coral", linestyle="--", label=f"Best ({best_threshold:.2f})")
axes[1].set_xlabel("Decision Threshold")
axes[1].set_ylabel("F1 Score")
axes[1].set_title("F1 Score vs Decision Threshold")
axes[1].legend()

plt.tight_layout()
plt.show()

# Compare default vs optimal threshold
print(f"=== Threshold Comparison ===")
print(f"\nDefault threshold (0.5):")
y_default = (y_bin_proba >= 0.5).astype(int)
print(classification_report(y_bin_test, y_default, target_names=["Normal", "Attack"]))

# Lower threshold for higher recall (SOC priority: catch all attacks)
print(f"Lower threshold (0.3) — Prioritise recall:")
y_low = (y_bin_proba >= 0.3).astype(int)
print(classification_report(y_bin_test, y_low, target_names=["Normal", "Attack"]))

## 14. Practical Scenario: Classifying New Flows

Simulate receiving new NetFlow records and classifying them in real time.

In [ ]:
# Simulate new incoming flows
new_flows = pd.DataFrame({
    "description": [
        "Web browsing (HTTPS)",
        "SSH session",
        "Suspicious: rapid SYN packets",
        "Large upload to external host",
        "DNS amplification attack",
        "Video streaming",
    ],
    "log_duration": [np.log1p(15), np.log1p(300), np.log1p(0.2), np.log1p(1800), np.log1p(2), np.log1p(60)],
    "bytes_per_packet": [800, 200, 60, 1400, 50, 1200],
    "log_packets_per_second": [np.log1p(10), np.log1p(3), np.log1p(500), np.log1p(5), np.log1p(10000), np.log1p(50)],
    "log_bytes_per_second": [np.log1p(8000), np.log1p(600), np.log1p(30000), np.log1p(7000), np.log1p(500000), np.log1p(60000)],
    "syn_ratio": [0.05, 0.01, 0.92, 0.005, 0.0, 0.02],
    "rst_ratio": [0.01, 0.0, 0.6, 0.0, 0.0, 0.01],
    "sent_recv_byte_ratio": [0.3, 1.5, 5.0, 80.0, 0.1, 0.2],
    "sent_recv_pkt_ratio": [0.8, 1.2, 10.0, 15.0, 50.0, 0.5],
    "protocol": [6, 6, 6, 6, 17, 6],
})

# Predict
X_new = new_flows[feature_cols]
predictions = pipe_multi.predict(X_new)
probabilities = pipe_multi.predict_proba(X_new)
max_proba = probabilities.max(axis=1)

results = pd.DataFrame({
    "Flow Description": new_flows["description"],
    "Predicted Class": predictions,
    "Confidence": [f"{p:.1%}" for p in max_proba],
    "Alert": ["🔴 HIGH" if p != "Normal" and conf > 0.7 else
              "🟡 MEDIUM" if p != "Normal" else
              "🟢 OK" for p, conf in zip(predictions, max_proba)]
})

print("=== Real-Time Flow Classification ===")
print(results.to_string(index=False))

# Detailed probabilities for suspicious flows
print("\n=== Probability Breakdown (suspicious flows) ===")
for i, desc in enumerate(new_flows["description"]):
    if predictions[i] != "Normal":
        probs = {cls: f"{p:.3f}" for cls, p in zip(classes, probabilities[i])}
        print(f"\n  {desc}:")
        print(f"    {probs}")

## 15. Limitations and Next Steps

In [ ]:
# Demonstrate the linear decision boundary limitation
from sklearn.inspection import DecisionBoundaryDisplay

# Use just 2 features for visualisation
X_2d = df[["log_packets_per_second", "syn_ratio"]].values
y_2d = LabelEncoder().fit_transform(df["label"])

scaler_2d = StandardScaler()
X_2d_scaled = scaler_2d.fit_transform(X_2d)

model_2d = LogisticRegression(solver="lbfgs",
                              max_iter=1000, random_state=42)
model_2d.fit(X_2d_scaled, y_2d)

# Plot decision boundaries
fig, ax = plt.subplots(figsize=(10, 6))
DecisionBoundaryDisplay.from_estimator(
    model_2d, X_2d_scaled, ax=ax,
    response_method="predict", alpha=0.2,
    cmap="Set3"
)

label_names = ["DDoS", "Exfiltration", "Normal", "Port Scan"]
scatter_colors = ["mediumpurple", "seagreen", "steelblue", "coral"]

for i, (name, color) in enumerate(zip(label_names, scatter_colors)):
    mask = y_2d == i
    ax.scatter(X_2d_scaled[mask, 0], X_2d_scaled[mask, 1],
             alpha=0.3, s=10, color=color, label=name)

ax.set_xlabel("Log Packets/sec (scaled)")
ax.set_ylabel("SYN Ratio (scaled)")
ax.set_title("Logistic Regression Decision Boundaries (2 features only)")
ax.legend()
plt.tight_layout()
plt.show()

print("Key observation: Decision boundaries are LINEAR (straight lines).")
print("This is both a strength (interpretable) and a limitation (can't capture complex patterns).")

## 16. Summary & Key Takeaways

| Aspect | Finding |
|:-------|:--------|
| **Sigmoid** | Maps linear score → probability [0, 1] |
| **Binary model** | Separates Normal vs Attack with high AUC |
| **Multi-class** | Softmax extends to 4 traffic types |
| **Coefficients** | SYN ratio and packet rate are strongest attack indicators |
| **Threshold** | Lower threshold → higher recall (catch more attacks, more FPs) |
| **Limitation** | Linear boundaries struggle with overlapping classes |

### Network Security Interpretation

- **Port Scan** signature: Very short flows + high SYN ratio + high RST ratio + tiny packets
- **DDoS** signature: Extreme packets/sec + moderate SYN + low bytes/packet
- **Exfiltration** signature: Long duration + large outbound ratio + large bytes/packet
- **Normal** traffic: Balanced send/receive, moderate duration, common ports

### Next Steps

- Try **Random Forest** or **Gradient Boosting** for non-linear boundaries
- Add more features: source IP entropy, flow inter-arrival time, port diversity
- Handle class imbalance (real attacks are rare) with SMOTE or class weights
- Use real NetFlow data (CIC-IDS2017, UNSW-NB15, or your own IPFIX collector)
- Deploy as a real-time classifier with Kafka + scikit-learn serving